# MVEmo: Music Emotion Recognition Multimodal (MoodNet: Audio + Lirik)

Notebook ini mengikuti paper **"A Multimodal Approach Towards Emotion Recognition of Music Using Audio and Lyrical Content"** (Bhattacharya & Kadambari, 2018), yaitu **MoodNet**:

- **Audio** -> log-mel-spectrogram 96x1366x1 -> Deep CNN 2D (conv 3x3 + max-pooling)
- **Lirik** -> matriks word embedding GloVe 100-d (baris lirik x kata x 100) -> Deep CNN 2D
- **Fusi** -> concatenation vektor hasil flatten kedua cabang -> dense 2048-1024-512-256 (ReLU, dropout 20%) -> softmax
- Varian **MoodNet-3/4/5** dan ablasi modalitas (audio saja, lirik saja, audio+lirik)

**Dataset:** `data/merge_bimodal_complete_metadata.csv` berisi 2.216 pasangan audio-lirik.
Lagu dengan audio < 29,12 detik (42 lagu) dibuang seluruhnya, termasuk liriknya. Data final yang dipakai: **2.174 lagu**.

**Fitur sudah diekstrak** oleh `scripts/prepare_features.py`:
- `processed/mel_spectrograms/{song_id}.npy` — log-mel 96x1366 dari MP3
- `processed/lyrics_embeddings/lyrics_glove_ids.npz` — indeks kata + matriks GloVe
- `processed/excluded_short_audio.csv` — daftar 42 lagu yang dibuang

Notebook ini **tidak melakukan ekstraksi fitur**. Alurnya: muat cache -> split `tvt_70_15_15` -> training MoodNet -> evaluasi.


In [ ]:
import os
import sys
from pathlib import Path

# Kurangi fragmentasi VRAM pada GPU kecil (harus diset sebelum torch/CUDA dipakai).
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")


def find_project_root(start=None):
    p = Path(start or Path.cwd()).resolve()
    for cand in [p, *p.parents]:
        if (cand / "data" / "merge_bimodal_complete_metadata.csv").exists():
            return cand
    return p


ROOT = find_project_root()
os.chdir(ROOT)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
print("Project root:", ROOT)


## 0. Dependensi dan prasyarat

Environment proyek ini dikelola dengan `uv`. Dari root proyek jalankan:

```powershell
python -m uv sync
python -m uv run python -m ipykernel install --user --name merge-bimodal --display-name "Python (MERGE Bimodal)"
```

Fitur audio dan lirik diekstrak sekali oleh script terpisah:

```powershell
python -m uv run python scripts/prepare_features.py
```

Script tersebut mendekode MP3 (librosa + soundfile), membuat log-mel, membangun token GloVe, dan menyimpan exclusion list. Notebook ini hanya memuat cache hasil script.


## 1. Konfigurasi


In [ ]:
import random
import re
import time
import warnings
from pathlib import Path

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")

# ---------- Path ----------
DATA = ROOT / "data"
META_CSV = DATA / "merge_bimodal_complete_metadata.csv"
AUDIO_DIR = DATA / "audio"
LYRICS_DIR = DATA / "lyrics"
SPLIT_NAME = "tvt_70_15_15"
SPLIT_DIR = DATA / "tvt_dataframes" / SPLIT_NAME

PROC = ROOT / "processed"
EXP = ROOT / "experiments"
CKPT = ROOT / "checkpoints"
MEL_DIR = PROC / "mel_spectrograms"
LYRICS_NPZ = PROC / "lyrics_embeddings" / "lyrics_glove_ids.npz"
EXCLUDED_CSV = PROC / "excluded_short_audio.csv"
for d in [PROC, PROC / "splits", EXP / "moodnet", CKPT]:
    d.mkdir(parents=True, exist_ok=True)

# ---------- Kolom metadata ----------
ID_KEY = "Audio_Song"
LYRIC_KEY = "Lyric_Song"
LABEL_KEY = "Quadrant"
CLASS_ORDER = ["Q1", "Q2", "Q3", "Q4"]

# ---------- Dimensi fitur dari cache ----------
MEL_BINS = 96
MEL_FRAMES = 1366

# ---------- Eksperimen ----------
SEEDS = [42, 43, 44]
USE_CLASS_WEIGHTS = False


## 2. Pemeriksaan GPU


In [ ]:
import torch

print("PyTorch:", torch.__version__)
print("CUDA tersedia:", torch.cuda.is_available())
if not torch.cuda.is_available():
    raise RuntimeError(
        "GPU CUDA tidak terdeteksi. Pasang PyTorch build CUDA dan pastikan driver NVIDIA aktif."
    )
DEVICE = torch.device("cuda:0")
props = torch.cuda.get_device_properties(0)
print(f"GPU: {props.name} | VRAM: {props.total_memory / 1024**3:.1f} GB | CUDA {torch.version.cuda}")
torch.backends.cudnn.benchmark = True


## 3. Inspeksi metadata CSV


In [ ]:
meta = pd.read_csv(META_CSV, dtype=str)
print("Jumlah baris:", len(meta))
print("Kolom:", list(meta.columns))
print(meta[[ID_KEY, LYRIC_KEY, LABEL_KEY, "Artist", "Title"]].head(10).to_string())
print("\nDistribusi Quadrant:")
print(meta[LABEL_KEY].value_counts().sort_index().to_string())


## 4. Manifest: pasangan audio/lirik + exclusion list

Pasangan diambil langsung dari kolom `Audio_Song`, `Lyric_Song`, dan `Quadrant`:

- Audio: `data/audio/{Quadrant}/{Audio_Song}.mp3`
- Lirik: `data/lyrics/{Quadrant}/{Lyric_Song}.txt`

Lagu dengan audio **< 29,12 detik** sudah diidentifikasi oleh `scripts/prepare_features.py` dan dicatat di `processed/excluded_short_audio.csv`. Notebook ini membuang lagu tersebut beserta liriknya tanpa mengukur ulang durasi MP3.


In [ ]:
required = [ID_KEY, LYRIC_KEY, LABEL_KEY]
missing_cols = [c for c in required if c not in meta.columns]
if missing_cols:
    raise ValueError(f"Kolom metadata tidak ditemukan: {missing_cols}")

man = pd.DataFrame({
    "song_id": meta[ID_KEY].astype(str).str.strip(),
    "lyric_id": meta[LYRIC_KEY].astype(str).str.strip(),
    "label_name": meta[LABEL_KEY].astype(str).str.strip(),
})
man["audio_path"] = [
    str(AUDIO_DIR / q / f"{s}.mp3")
    for q, s in zip(man.label_name, man.song_id)
]
man["lyrics_path"] = [
    str(LYRICS_DIR / q / f"{s}.txt")
    for q, s in zip(man.label_name, man.lyric_id)
]
man["has_audio"] = [Path(p).exists() for p in man.audio_path]
man["has_lyrics"] = [Path(p).exists() for p in man.lyrics_path]

print(f"Audio ditemukan : {man.has_audio.sum()} / {len(man)}")
print(f"Lirik ditemukan : {man.has_lyrics.sum()} / {len(man)}")
if not (man.has_audio & man.has_lyrics).all():
    print("Contoh file yang hilang:")
    print(man.loc[~(man.has_audio & man.has_lyrics),
                  ["song_id", "lyric_id", "audio_path", "lyrics_path"]].head().to_string())

man = man[man.has_audio & man.has_lyrics].copy()

# Daftar lagu < 29,12 dtk dibuat oleh scripts/prepare_features.py.
if not EXCLUDED_CSV.exists():
    raise FileNotFoundError(
        f"{EXCLUDED_CSV} tidak ditemukan. Jalankan: "
        "python -m uv run python scripts/prepare_features.py"
    )
excluded = pd.read_csv(EXCLUDED_CSV, dtype=str)
if "song_id" not in excluded.columns:
    raise ValueError(f"Kolom song_id tidak ada di {EXCLUDED_CSV}")
excluded_ids = set(excluded.song_id.astype(str).str.strip())

unknown_ex = excluded_ids - set(man.song_id)
if unknown_ex:
    raise ValueError(f"{len(unknown_ex)} ID exclusion tidak ada di metadata: "
                     f"{sorted(unknown_ex)[:5]}")

man = man[~man.song_id.isin(excluded_ids)].copy()
print(f"\nDibuang sesuai {EXCLUDED_CSV.name}: {len(excluded_ids)} lagu")
if "label_name" in excluded.columns:
    print(excluded.label_name.value_counts().sort_index().to_string())

unknown = sorted(set(man.label_name) - set(CLASS_ORDER))
if unknown:
    raise ValueError(f"Label tidak dikenal: {unknown}")
man["label_idx"] = man.label_name.map({c: i for i, c in enumerate(CLASS_ORDER)})
NUM_CLASSES = len(CLASS_ORDER)

dup = man.song_id.duplicated().sum()
if dup:
    raise ValueError(f"{dup} song_id duplikat di metadata")

print(f"\nTotal lagu dipakai: {len(man)}")
print("\nDistribusi kelas (seluruh data):")
print(man.label_name.value_counts().sort_index().to_string())


## 5. Split train/validation/test dari file yang disediakan

Split tidak dibuat ulang. File `tvt_70_15_15_{train,validate,test}_bimodal_complete.csv` dipetakan berdasarkan `Song`, lalu lagu yang sudah difilter otomatis tidak ikut.


In [ ]:
split_frames = []
for part, fname in [("train", "train"), ("validation", "validate"), ("test", "test")]:
    p = SPLIT_DIR / f"{SPLIT_NAME}_{fname}_bimodal_complete.csv"
    if not p.exists():
        raise FileNotFoundError(p)
    df = pd.read_csv(p, dtype=str)
    df.columns = [c.strip() for c in df.columns]
    if not {"Song", "Quadrant"}.issubset(df.columns):
        raise ValueError(f"Kolom split tidak sesuai: {list(df.columns)} di {p}")
    df = df.rename(columns={"Song": "song_id", "Quadrant": "label_name"})
    df["split"] = part
    split_frames.append(df[["song_id", "label_name", "split"]])

splits = pd.concat(split_frames, ignore_index=True)
if splits.song_id.duplicated().any():
    raise ValueError("song_id duplikat antar-file split")

split_map = dict(zip(splits.song_id, splits.split))
quad_map = dict(zip(splits.song_id, splits.label_name))

man["split"] = man.song_id.map(split_map)
missing_split = man[man.split.isna()]
if len(missing_split):
    raise ValueError(f"{len(missing_split)} lagu tidak ada di file split: "
                     f"{missing_split.song_id.head().tolist()}")

bad_quad = man[man.label_name != man.song_id.map(quad_map)]
if len(bad_quad):
    raise ValueError(f"{len(bad_quad)} label tidak cocok dengan file split")

print("Distribusi split x kelas:")
print(pd.crosstab(man.split, man.label_name).to_string())
print("\nJumlah per split:")
print(man.split.value_counts().reindex(["train", "validation", "test"]).to_string())

man.to_csv(PROC / "manifest.csv", index=False)
for s in ["train", "validation", "test"]:
    man[man.split == s].to_csv(PROC / "splits" / f"{s}.csv", index=False)


## 6. Muat fitur dari cache (audio + lirik)

Tidak ada ekstraksi di notebook. Semua fitur sudah dibuat oleh `scripts/prepare_features.py`:

- `processed/mel_spectrograms/{song_id}.npy` — log-mel `96x1366`
- `processed/lyrics_embeddings/lyrics_glove_ids.npz` — `ids`, `TOK`, `EMB`, `txt_avail`

Sel ini memuat cache, menyelaraskan urutannya dengan manifest, lalu membentuk tensor siap training.


In [ ]:
if not MEL_DIR.is_dir():
    raise FileNotFoundError(
        f"Cache mel tidak ditemukan: {MEL_DIR}. Jalankan: "
        "python -m uv run python scripts/prepare_features.py"
    )
if not LYRICS_NPZ.exists():
    raise FileNotFoundError(
        f"Cache lirik tidak ditemukan: {LYRICS_NPZ}. Jalankan: "
        "python -m uv run python scripts/prepare_features.py"
    )

z = np.load(LYRICS_NPZ, allow_pickle=True)
lyr_ids = z["ids"].astype(str)
lut = {sid: i for i, sid in enumerate(lyr_ids)}
missing_lyr = [s for s in man.song_id if s not in lut]
if missing_lyr:
    raise ValueError(f"{len(missing_lyr)} lagu tidak ada di cache lirik: "
                     f"{missing_lyr[:5]}")
sel = np.array([lut[s] for s in man.song_id])
TOK = z["TOK"][sel]
EMB_MATRIX = z["EMB"]
if "txt_avail" in z.files:
    TXT_AVAIL = z["txt_avail"][sel].astype(bool)
else:
    TXT_AVAIL = (TOK.reshape(len(TOK), -1) != 0).any(axis=1)
print(f"Cache lirik: {TOK.shape[0]} lagu | EMB_MATRIX {EMB_MATRIX.shape}")

MEL_AVAIL = np.array([(MEL_DIR / f"{s}.npy").exists() for s in man.song_id])
MN_MASK = MEL_AVAIL & TXT_AVAIL
print(f"Mel tersedia={MEL_AVAIL.sum()} | lirik tersedia={TXT_AVAIL.sum()} | "
      f"irisan={MN_MASK.sum()}")
if not MN_MASK.all():
    print(man.loc[~MN_MASK, ["song_id", "split", "label_name"]].head(20).to_string())
    raise RuntimeError(
        "Cache tidak lengkap. Jalankan ulang: "
        "python -m uv run python scripts/prepare_features.py"
    )

MEL_ALL = np.zeros((len(man), MEL_BINS, MEL_FRAMES), dtype=np.float16)
for i, s in enumerate(man.song_id):
    MEL_ALL[i] = np.load(MEL_DIR / f"{s}.npy")

tr_idx = np.where(MN_MASK & (man.split.values == "train"))[0]
if len(tr_idx) == 0:
    raise RuntimeError("Tidak ada data train yang lengkap")
_s = _s2 = _n = 0.0
for i in tr_idx:
    x = MEL_ALL[i].astype(np.float64)
    _s += x.sum()
    _s2 += (x ** 2).sum()
    _n += x.size
MEL_MU = _s / _n
MEL_SD = float(np.sqrt(_s2 / _n - MEL_MU ** 2)) + 1e-6
print(f"Mel mean={MEL_MU:.2f} std={MEL_SD:.2f} (dari train)")

MEL_T = torch.from_numpy(
    ((MEL_ALL.astype(np.float32) - MEL_MU) / MEL_SD).astype(np.float16)
).to(DEVICE)
TOK_T = torch.from_numpy(TOK).long().to(DEVICE)
Y_ALL = man.label_idx.values
del MEL_ALL
print("MEL_T:", tuple(MEL_T.shape), "| TOK_T:", tuple(TOK_T.shape))
print("\nSubset lengkap per split:")
print(man[MN_MASK].groupby("split").size().reindex(["train", "validation", "test"]).to_string())


## 7. Arsitektur MoodNet dan fungsi training

- **Cabang CNN**: `Conv 3x3 -> ReLU -> MaxPool`, filter 128-256-512-1024-2048. Pooling audio `(2,4),(2,4),(2,4),(3,5),(4,4)` dan lirik `(2,2),(2,2),(2,2),(3,2),(4,2)`.
- **Fusi**: concatenation -> Dense 2048 -> 1024 -> 512 -> 256 (ReLU, dropout 20%) -> softmax.
- Optimizer **Adam**, loss **categorical cross-entropy**, early stopping pada macro-F1 validation.

Hanya MoodNet-5 yang menghasilkan vektor 2048 per cabang seperti paper; MoodNet-3/4 memakai global max-pooling sebelum flatten.

Batch training otomatis memakai 8 untuk MoodNet-3/4 dan 4 untuk MoodNet-5 (parameter lebih besar); batch prediksi 32/8. Jika muncul peringatan OOM dari CUDA allocator, PyTorch biasanya mencoba ulang; turunkan batch jika training benar-benar gagal.


In [ ]:
import copy
import torch.nn as nn
import torch.nn.functional as F
from sklearn.metrics import (
    accuracy_score, confusion_matrix, f1_score, precision_score, recall_score,
)

CHANNELS = [128, 256, 512, 1024, 2048]
AUDIO_POOLS = [(2, 4), (2, 4), (2, 4), (3, 5), (4, 4)]
TEXT_POOLS = [(2, 2), (2, 2), (2, 2), (3, 2), (4, 2)]


def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


class ConvBranch(nn.Module):
    def __init__(self, in_ch, n_layers, pools):
        super().__init__()
        layers, c = [], in_ch
        for i in range(n_layers):
            layers += [
                nn.Conv2d(c, CHANNELS[i], kernel_size=3, padding=1),
                nn.ReLU(inplace=True),
                nn.MaxPool2d(pools[i]),
            ]
            c = CHANNELS[i]
        self.features = nn.Sequential(*layers)
        self.out_dim = c
        for m in self.modules():
            if isinstance(m, nn.Conv2d):
                nn.init.kaiming_normal_(m.weight, nonlinearity="relu")
                nn.init.zeros_(m.bias)

    def forward(self, x):
        return F.adaptive_max_pool2d(self.features(x), 1).flatten(1)


class MoodNet(nn.Module):
    def __init__(self, n_layers, n_classes, mods, emb_matrix, dropout=0.2):
        super().__init__()
        self.mods = list(mods)
        d = 0
        if "audio" in self.mods:
            self.audio = ConvBranch(1, n_layers, AUDIO_POOLS)
            d += self.audio.out_dim
        if "lyrics" in self.mods:
            self.emb = nn.Embedding.from_pretrained(
                torch.as_tensor(emb_matrix), freeze=True, padding_idx=0
            )
            self.text = ConvBranch(emb_matrix.shape[1], n_layers, TEXT_POOLS)
            d += self.text.out_dim
        dims, head = [d, 2048, 1024, 512, 256], []
        for a, b in zip(dims[:-1], dims[1:]):
            head += [nn.Linear(a, b), nn.ReLU(inplace=True), nn.Dropout(dropout)]
        head.append(nn.Linear(256, n_classes))
        self.head = nn.Sequential(*head)

    def forward(self, mel=None, tok=None):
        z = []
        if "audio" in self.mods:
            z.append(self.audio(mel))
        if "lyrics" in self.mods:
            z.append(self.text(self.emb(tok).permute(0, 3, 1, 2)))
        return self.head(torch.cat(z, 1))


MN_HP = dict(lr=1e-4, batch_size=8, epochs=60, patience=10, dropout=0.2)


def metrics(y, p):
    return dict(
        accuracy=accuracy_score(y, p),
        precision_macro=precision_score(y, p, average="macro", zero_division=0),
        recall_macro=recall_score(y, p, average="macro", zero_division=0),
        macro_f1=f1_score(y, p, average="macro", zero_division=0),
        weighted_f1=f1_score(y, p, average="weighted", zero_division=0),
    )


def run_moodnet(mods, n_layers, seed, ckpt_path=None):
    set_seed(seed)
    torch.cuda.empty_cache()
    rows = {s: np.where(MN_MASK & (man.split.values == s))[0]
            for s in ["train", "validation", "test"]}
    idx = {s: torch.tensor(r, device=DEVICE) for s, r in rows.items()}
    Y = torch.tensor(Y_ALL, dtype=torch.long, device=DEVICE)

    model = MoodNet(n_layers, NUM_CLASSES, mods, EMB_MATRIX,
                    MN_HP["dropout"]).to(DEVICE)
    opt = torch.optim.Adam(
        [p for p in model.parameters() if p.requires_grad], lr=MN_HP["lr"]
    )
    if USE_CLASS_WEIGHTS:
        counts = np.bincount(Y_ALL[rows["train"]], minlength=NUM_CLASSES).astype(np.float64)
        weights = counts.sum() / (NUM_CLASSES * np.maximum(counts, 1.0))
        loss_fn = nn.CrossEntropyLoss(
            weight=torch.tensor(weights, dtype=torch.float32, device=DEVICE)
        )
    else:
        loss_fn = nn.CrossEntropyLoss()
    scaler = torch.amp.GradScaler("cuda")

    def get_batch(ix):
        mel = MEL_T[ix].unsqueeze(1) if "audio" in mods else None
        tok = TOK_T[ix] if "lyrics" in mods else None
        return mel, tok

    @torch.no_grad()
    def predict(split):
        model.eval()
        out = []
        pbs = 8 if n_layers >= 5 else 32
        for i in range(0, len(idx[split]), pbs):
            mel, tok = get_batch(idx[split][i:i + pbs])
            with torch.autocast("cuda", dtype=torch.float16):
                out.append(model(mel, tok).argmax(1).cpu())
        return torch.cat(out).numpy()

    n = len(idx["train"])
    bs = 4 if n_layers >= 5 else MN_HP["batch_size"]
    y_val = Y[idx["validation"]].cpu().numpy()
    best_f1, best_state, bad = -1.0, None, 0
    for epoch in range(MN_HP["epochs"]):
        model.train()
        perm = idx["train"][torch.randperm(n, device=DEVICE)]
        for i in range(0, n, bs):
            b = perm[i:i + bs]
            mel, tok = get_batch(b)
            opt.zero_grad(set_to_none=True)
            with torch.autocast("cuda", dtype=torch.float16):
                loss = loss_fn(model(mel, tok), Y[b])
            scaler.scale(loss).backward()
            scaler.step(opt)
            scaler.update()
        f1 = f1_score(y_val, predict("validation"), average="macro", zero_division=0)
        if f1 > best_f1:
            best_f1, bad = f1, 0
            best_state = {k: v.detach().cpu().clone()
                          for k, v in model.state_dict().items()}
        else:
            bad += 1
            if bad >= MN_HP["patience"]:
                break
    model.load_state_dict(best_state)
    if ckpt_path is not None:
        torch.save(best_state, ckpt_path)

    y_true = Y[idx["test"]].cpu().numpy()
    y_pred = predict("test")
    res = metrics(y_true, y_pred)
    res.update(val_macro_f1=best_f1, epochs_run=epoch + 1,
               n_train=len(rows["train"]), n_val=len(rows["validation"]),
               n_test=len(rows["test"]))
    del model, opt
    torch.cuda.empty_cache()
    return res, y_true, y_pred


for nl in (3, 4, 5):
    _m = MoodNet(nl, NUM_CLASSES, ["audio", "lyrics"], EMB_MATRIX).to(DEVICE).eval()
    with torch.no_grad(), torch.autocast("cuda", dtype=torch.float16):
        _o = _m(MEL_T[:2].unsqueeze(1), TOK_T[:2])
    print(f"MoodNet-{nl}: flatten audio={_m.audio.out_dim}, lirik={_m.text.out_dim}, "
          f"keluaran={tuple(_o.shape)}, parameter terlatih="
          f"{sum(p.numel() for p in _m.parameters() if p.requires_grad) / 1e6:.1f} M")
    del _m
torch.cuda.empty_cache()


## 8. Jalankan eksperimen

MoodNet-3/4/5 dengan fusi audio+lirik, dan ablasi modalitas pada MoodNet-4: audio saja, lirik saja, audio+lirik.


In [ ]:
EXPERIMENTS = {
    "MoodNet-3 | audio+lyrics": (["audio", "lyrics"], 3),
    "MoodNet-4 | audio+lyrics": (["audio", "lyrics"], 4),
    "MoodNet-5 | audio+lyrics": (["audio", "lyrics"], 5),
    "MoodNet-4 | audio": (["audio"], 4),
    "MoodNet-4 | lyrics": (["lyrics"], 4),
}

all_rows, preds = [], {}
t0 = time.time()
for name, (mods, n_layers) in EXPERIMENTS.items():
    slug = re.sub(r"[^A-Za-z0-9]+", "_", name).strip("_")
    folder = EXP / "moodnet" / slug
    folder.mkdir(parents=True, exist_ok=True)
    for seed in SEEDS:
        ckpt_path = CKPT / f"{slug}_seed{seed}.pt"
        res, yt, yp = run_moodnet(mods, n_layers, seed, ckpt_path=ckpt_path)
        res.update(experiment=name, architecture=f"MoodNet-{n_layers}",
                   modalities="+".join(mods), seed=seed)
        all_rows.append(res)
        preds[(name, seed)] = (yt, yp)
        np.savez(folder / f"preds_seed{seed}.npz", y_true=yt, y_pred=yp)
        print(f"{name:26s} seed={seed} acc={res['accuracy']:.4f} "
              f"macroF1={res['macro_f1']:.4f} "
              f"(train={res['n_train']}, test={res['n_test']}, "
              f"epoch={res['epochs_run']})")
    pd.DataFrame([r for r in all_rows if r["experiment"] == name]).to_json(
        folder / "metrics.json", orient="records", indent=2
    )
print(f"\nSelesai dalam {(time.time() - t0) / 60:.1f} menit")


## 9. Ringkasan hasil dan visualisasi


In [ ]:
import matplotlib.pyplot as plt

res_df = pd.DataFrame(all_rows)
metric_cols = ["accuracy", "precision_macro", "recall_macro",
               "macro_f1", "weighted_f1"]
summary = (res_df.groupby("experiment", sort=False)[metric_cols]
           .agg(["mean", "std"]).round(4))
summary[("n_train", "")] = res_df.groupby("experiment", sort=False)["n_train"].first()
summary[("n_test", "")] = res_df.groupby("experiment", sort=False)["n_test"].first()
res_df.to_csv(EXP / "moodnet" / "results_all_runs.csv", index=False)
summary.to_csv(EXP / "moodnet" / "results_summary.csv")

fig, ax = plt.subplots(figsize=(10, 4))
order = list(EXPERIMENTS.keys())
means = summary.loc[order, ("macro_f1", "mean")]
stds = summary.loc[order, ("macro_f1", "std")]
ax.barh(range(len(order)), means.values, xerr=stds.values,
        color="#4C78A8", capsize=3)
ax.set_yticks(range(len(order)))
ax.set_yticklabels(order)
ax.invert_yaxis()
ax.set_xlabel("Macro-F1 test (mean +- std antar seed)")
ax.set_title("MoodNet pada 2.174 lagu")
ax.grid(axis="x", alpha=0.3)
fig.tight_layout()
fig.savefig(EXP / "moodnet" / "macro_f1_summary.png", dpi=160)
plt.show()

cm_name = "MoodNet-4 | audio+lyrics"
cm_seed = SEEDS[0]
if (cm_name, cm_seed) in preds:
    yt, yp = preds[(cm_name, cm_seed)]
    cm = confusion_matrix(yt, yp, labels=list(range(NUM_CLASSES)))
    fig, ax = plt.subplots(figsize=(4.5, 4))
    im = ax.imshow(cm, cmap="Blues")
    ax.set_xticks(range(NUM_CLASSES))
    ax.set_yticks(range(NUM_CLASSES))
    ax.set_xticklabels(CLASS_ORDER)
    ax.set_yticklabels(CLASS_ORDER)
    ax.set_xlabel("Prediksi")
    ax.set_ylabel("Aktual")
    ax.set_title(f"Confusion matrix {cm_name} seed={cm_seed}")
    for i in range(NUM_CLASSES):
        for j in range(NUM_CLASSES):
            ax.text(j, i, cm[i, j], ha="center", va="center",
                    color="white" if cm[i, j] > cm.max() / 2 else "black")
    fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
    fig.tight_layout()
    fig.savefig(EXP / "moodnet" / "confusion_moodnet4_fusion.png", dpi=160)
    plt.show()

summary


## Catatan interpretasi

- Fitur audio dan lirik sudah diekstrak oleh `scripts/prepare_features.py`; notebook ini hanya memuat cache, tidak mendekode MP3 atau membangun GloVe.
- Dataset final: **2.174 lagu** dari 2.216 pasangan; 42 lagu berdurasi < 29,12 dtk dibuang beserta liriknya dan dicatat di `processed/excluded_short_audio.csv`.
- Split mengikuti `tvt_70_15_15` yang disediakan; train/validation/test = 1.524/325/325.
- Periksa `processed/mel_failed.csv` (dibuat script) jika ada audio yang gagal diekstrak.
- GloVe hanya bahasa Inggris; kata OOV menjadi vektor nol.
- Kelas tidak seimbang (Q2 lebih banyak). Metrik utama macro-F1; `USE_CLASS_WEIGHTS = True` dapat dicoba.
- Split disediakan apa adanya; potensi versi lagu yang sama antar-split tidak dianalisis ulang.
- Training penuh 5 eksperimen x 3 seed dapat memakan waktu lama di GPU 6 GB. Kurangi `SEEDS` atau `EXPERIMENTS` untuk uji cepat.
